# 02. GARCH(1,1): fit, forecast, and an honest verdict

Fit GARCH(1,1) on the first 70% of the sample, produce one-day-ahead variance
forecasts over the last 30% with parameters frozen at their train estimates, and
score them against two naive benchmarks under QLIKE (primary) and MSE
(secondary).

At a one-day horizon the expected result is undramatic: GARCH should beat the
unconditional benchmark clearly and naive persistence modestly, if at all. The
aim is to measure that cleanly rather than to make the model look good.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from arch import arch_model

from garch_lab.data import load_prices
from garch_lab.returns import log_returns
from garch_lab.volatility import rolling_vol
from garch_lab.evaluation import (
    chronological_split, mse, qlike, rw_variance_forecast,
)

import os
path_here = os.getcwd()
project_path = os.path.dirname(path_here)
os.chdir(project_path)

px = load_prices("^GSPC", start="2005-01-01")

END_DATE = "2026-08-03"  # same frozen end date as notebook 01
px = px[px.index <= END_DATE]

r = log_returns(px)

## The split comes first

Evaluation design is decided before the model. Deciding it after seeing forecasts
is how look-ahead seeps in.

A random train/test split is not merely suboptimal for serially dependent data,
it is invalid: neighbouring days are strongly dependent, so a randomly held-out
day sits between two days the model has already trained on, and its
neighbourhood leaks into training. From here every quantity carries a mental
timestamp of when it became knowable, which rules out any full-sample statistic
as a modelling input.

The four functions used below live in
[src/garch_lab/evaluation.py](../src/garch_lab/evaluation.py) and need no model,
so they are written and tested before anything is fitted.

In [2]:
from garch_lab.evaluation import chronological_split

r_train, r_test = chronological_split(r, 0.7)

split_date = r_test.index[0]

print(f"Train set: {r_train.index[0]} to {r_train.index[-1]}")
print(f"Test set: {r_test.index[0]} to {r_test.index[-1]}")
print(f"Test set length: {len(r_test)}")

Train set: 2005-01-04 00:00:00 to 2020-02-06 00:00:00
Test set: 2020-02-07 00:00:00 to 2026-08-03 00:00:00
Test set length: 1629


Train and test meet on adjacent trading days, and the test window holds 1,629
observations.

Notebook 01 computed full-sample statistics: the mean, variance, kurtosis and
skewness of the returns. That is acceptable for description, since it summarizes
the whole dataset, but it would not be valid as a modelling input, because it
uses information from the test period that would not have been available at
training time.

## Fitting GARCH(1,1)

**GARCH(1,1).** Returns are $r_t = \sigma_t z_t$ with $z_t \overset{iid}{\sim} (0,1)$,
and the conditional variance follows

$$
\sigma_t^2 = \omega + \alpha\, \varepsilon_{t-1}^2 + \beta\, \sigma_{t-1}^2,
$$

with $\omega > 0$, $\alpha, \beta \ge 0$, and covariance-stationarity requiring
$\alpha + \beta < 1$, in which case the unconditional variance is
$\sigma^2 = \omega / (1 - \alpha - \beta)$.

Read the variance equation as a baseline ($\omega$), a reaction to news
($\alpha$, the weight on yesterday's squared shock) and a memory term ($\beta$,
the weight on yesterday's variance).

Returns are scaled by 100 before fitting. A likelihood over variances of order
$10^{-4}$ is badly conditioned for the optimizer, and `arch` warns about it.

Both fits use the train window only, via `last_obs`, so the parameters are
estimated without seeing a single test observation.

In [3]:
garch_n = arch_model(100*r, vol="Garch", p=1, q=1, dist="normal")
garch_t = arch_model(100*r, vol="Garch", p=1, q=1, dist="t")

garch_model_n = garch_n.fit(disp="off", last_obs=r_train.index[-1])
garch_model_t = garch_t.fit(disp="off", last_obs=r_train.index[-1])

assert not garch_model_n.convergence_flag, "GARCH(1,1)-N did not converge"
assert not garch_model_t.convergence_flag, "GARCH(1,1)-t did not converge"

print("With Normal distribution:")
print(garch_model_n.summary())
print("\n")
print("Omega is positive {:.4f}".format(garch_model_n.params["omega"]))
print("Alpha is positive {:.4f}".format(garch_model_n.params["alpha[1]"]))
print("Beta is positive {:.4f}".format(garch_model_n.params["beta[1]"]))
print("Alpha + Beta is less than 1 {:.4f}".format(garch_model_n.params["alpha[1]"] + garch_model_n.params["beta[1]"]))
print("Implied unconditional volatility is {:.4f}".format(np.sqrt(252) * np.sqrt(garch_model_n.params["omega"] / (1 - garch_model_n.params["alpha[1]"] - garch_model_n.params["beta[1]"]))))

print("With Student's t-distribution:")
print(garch_model_t.summary())
print("\n")
print("Omega is positive {:.4f}".format(garch_model_t.params["omega"]))
print("Alpha is positive {:.4f}".format(garch_model_t.params["alpha[1]"]))
print("Beta is positive {:.4f}".format(garch_model_t.params["beta[1]"]))
print("Alpha + Beta is less than 1 {:.4f}".format(garch_model_t.params["alpha[1]"] + garch_model_t.params["beta[1]"]))
print("Implied unconditional volatility is {:.4f}".format(np.sqrt(252) * np.sqrt(garch_model_t.params["omega"] / (1 - garch_model_t.params["alpha[1]"] - garch_model_t.params["beta[1]"]))))

With Normal distribution:
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -4883.80
Distribution:                  Normal   AIC:                           9775.61
Method:            Maximum Likelihood   BIC:                           9800.58
                                        No. Observations:                 3798
Date:                Wed, Aug 05 2026   Df Residuals:                     3797
Time:                        13:09:39   Df Model:                            1
                                 Mean Model                                 
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
mu             0.0665  1.198e-02

The Gaussian fit gives $\alpha + \beta = 0.9775$ and an implied unconditional
volatility of 16.6% annualized. The Student-t fit is far more persistent at
$\alpha + \beta = 0.9960$, and its implied unconditional volatility of 31% sits
well above anything the training sample realized.

On why the conditional $\hat{\nu}$ is larger than notebook 01's unconditional
one: the generated Gaussians have a different random variance each period, and
mixing them produces heavier tails than any single Gaussian. Conditioning on
$\sigma_t$ absorbs that part, so the tails left for the innovation distribution to
explain are lighter and $\hat{\nu}$ comes out larger.

The log-likelihood is scale-dependent, so comparing one from a fit on 100·r with
one from a fit on raw r would be meaningless.

## In-sample sanity: the filter view

GARCH is the boxcar of notebook 01 upgraded to a learned exponential filter.
Overlaying the fitted conditional volatility with the 21-day rolling volatility
shows the difference in impulse response before the model is trusted out of
sample.

`res.conditional_volatility` is daily and in percent, because the fit used
100·r, while `rolling_vol` on raw `r` is daily in raw units. Both are converted
to annualized percent below, or the plot would show the ×100 and ×15.9 tells.

The train/test divider uses `.to_pydatetime()`: a bare `pd.Timestamp` renders in
`fig.show()` but crashes `fig.write_image()`, because kaleido serializes with
orjson, which cannot encode it. This only affects scalar dates in the layout.

In [4]:
annualized_conditional_vol_n = np.sqrt(252) * garch_model_n.conditional_volatility
rolling_21d_vol = rolling_vol(r, window=21)*100*np.sqrt(252)

fig = go.Figure()
fig.add_trace(go.Scatter(x=annualized_conditional_vol_n.index, y=annualized_conditional_vol_n, mode='lines', name='Annualized Conditional Volatility (GARCH(1,1)-N)'))
fig.add_trace(go.Scatter(x=rolling_21d_vol.index, y=rolling_21d_vol, mode='lines', name='21-Day Rolling Volatility'))
fig.add_vline(x=split_date.to_pydatetime(), line_dash='dash')
fig.update_layout(title='Annualized Conditional Volatility vs 21-Day Rolling Volatility',
                  xaxis_title='Date',
                  yaxis_title='Volatility',
                  legend=dict(x=0, y=1, traceorder='normal'),
                  template='plotly_white')

fig.show()

fig.write_image("figures/fig4_rolling_vs_garch.png")

The fitted filter has a memory that lets old events fade out, instead of
weighting the whole window equally and then dropping it sharply. GARCH also
reverts to the unconditional variance smoothly, which the boxcar cannot do.

## One-step out-of-sample forecasts, and the alignment discipline

The forecast made on day $t$ targets day $t+1$. In a series indexed by target
date it must therefore occupy row $t+1$, which is one shift, and
`assert h.index.equals(p.index)` is the line that proves the alignment.

Below: one-step-ahead GARCH variance forecasts over the whole test window for
both fits, the proxy $p = (100 \cdot r_{test})^2$, and both benchmarks. The
parameters stay frozen at their train estimates while the $\sigma^2$ recursion
rolls through the observed test returns, so no refit loop is needed.

Everything is fed the same percent-scaled returns and stays in %²-per-day units,
with no square roots and no 252 until a plot axis needs them.

In [5]:
# Forecast of the Gaussian GARCH model
forecast_n = garch_model_n.forecast(start=r_train.index[-1], horizon=1, reindex=False)
h_garch_n = forecast_n.variance["h.1"].shift(1)  # Shift to align with the test index
h_garch_n = h_garch_n.loc[r_test.index]  # Cut to the test index

# Forecast of the Student-t GARCH model
forecast_t = garch_model_t.forecast(start=r_train.index[-1], horizon=1, reindex=False)
h_garch_t = forecast_t.variance["h.1"].shift(1)  # Shift to align with the test index
h_garch_t = h_garch_t.loc[r_test.index]  # Cut to the test index

# Baseline 1: the random-walk variance forecast, i.e. the mean of r^2 over the
# 21 days strictly before t. It is a variance, and it excludes t by construction.
h_rw = rw_variance_forecast(r*100, window=21)
h_rw = h_rw.loc[r_test.index] # Cut to the test index

# Baseline 2: the unconditional variance forecast, i.e. the training-sample mean
# of squared returns, held constant across the test window.
training_variance = np.mean((r_train*100)**2)
h_unc = training_variance * (r_test*0 + 1)

# Proxy
p = (100*r_test)**2


# Plot the forecasts
fig = go.Figure()
fig.add_trace(go.Scatter(x=h_garch_n.index, y=h_garch_n, mode='lines', name='GARCH(1,1)-N Forecast'))
fig.add_trace(go.Scatter(x=h_garch_t.index, y=h_garch_t, mode='lines', name='GARCH(1,1)-t Forecast'))
fig.add_trace(go.Scatter(x=h_rw.index, y=h_rw, mode='lines', name='Random Walk Forecast'))
fig.add_trace(go.Scatter(x=h_unc.index, y= h_unc, mode='lines', name='Unconditional Forecast'))
fig.add_trace(go.Scatter(x=p.index, y=p, mode='lines', name='Proxy (Squared Returns)'))

fig.update_layout(title='One-day-ahead variance forecasts vs. squared-return proxy',
                  xaxis_title='Date',
                  yaxis_title='Variance (%² per day)',
                  legend=dict(x=0, y=1, traceorder='normal'),
                  template='plotly_white')
fig.show()

assert h_garch_n.index.equals(p.index)
assert h_garch_t.index.equals(p.index)
assert h_rw.index.equals(p.index)
assert h_unc.index.equals(p.index)

print("Mean of GARCH(1,1)-N Forecast:", np.mean(h_garch_n))
print("Mean of GARCH(1,1)-t Forecast:", np.mean(h_garch_t))
print("Mean of Random Walk Forecast:", np.mean(h_rw))
print("Mean of Unconditional Forecast:", np.mean(h_unc))
print("Mean of Proxy (Squared Returns):", np.mean(p))

Mean of GARCH(1,1)-N Forecast: 1.589370943498399
Mean of GARCH(1,1)-t Forecast: 1.7392783252639716
Mean of Random Walk Forecast: 1.678111034881204
Mean of Unconditional Forecast: 1.3420699368429132
Mean of Proxy (Squared Returns): 1.6773215063884457


In [6]:
# Perturbation self check
# We multiply a mid-test volatility by 10 and check that the forecasts change accordingly
mid_test_index = r_test.index[len(r_test)//2]
r_perturbed = r.copy()
r_perturbed.loc[mid_test_index] *= 10

garch_model_n_perturbed = arch_model(100*r_perturbed, vol="Garch", p=1, q=1, dist="normal").fit(disp="off", last_obs=r_train.index[-1])
forecast_n_perturbed = garch_model_n_perturbed.forecast(start=r_train.index[-1], horizon=1, reindex=False)
h_garch_n_perturbed = forecast_n_perturbed.variance["h.1"].shift(1)  # Shift to align with the test index
h_garch_n_perturbed = h_garch_n_perturbed.loc[r_test.index]  # Cut to the test index

garch_model_t_perturbed = arch_model(100*r_perturbed, vol="Garch", p=1, q=1, dist="t").fit(disp="off", last_obs=r_train.index[-1])
forecast_t_perturbed = garch_model_t_perturbed.forecast(start=r_train.index[-1], horizon=1, reindex=False)
h_garch_t_perturbed = forecast_t_perturbed.variance["h.1"].shift(1)  # Shift to align with the test index
h_garch_t_perturbed = h_garch_t_perturbed.loc[r_test.index]  # Cut to the test index

# Plot perturbed vs unperturbed forecasts
fig = go.Figure()
fig.add_trace(go.Scatter(x=h_garch_n.index, y=h_garch_n, mode='lines', name='GARCH(1,1)-N Forecast'))
fig.add_trace(go.Scatter(x=h_garch_n_perturbed.index, y=h_garch_n_perturbed, mode='lines', name='GARCH(1,1)-N Perturbed Forecast'))
fig.add_trace(go.Scatter(x=h_garch_t.index, y=h_garch_t, mode='lines', name='GARCH(1,1)-t Forecast'))
fig.add_trace(go.Scatter(x=h_garch_t_perturbed.index, y=h_garch_t_perturbed, mode='lines', name='GARCH(1,1)-t Perturbed Forecast'))
fig.update_layout(title='One-day-ahead variance forecasts vs. perturbed forecasts',
                  xaxis_title='Date',
                  yaxis_title='Variance (%² per day)',
                  legend=dict(x=0, y=1, traceorder='normal'),
                  template='plotly_white')
fig.show()

assert np.allclose(garch_model_n.params, garch_model_n_perturbed.params)

The first forecast targets the first test day and the index assertions pass.

The perturbation check above multiplies one mid-test return by 10 and rebuilds
the forecast series the same way. The forecast for that date is unchanged while
later ones move, which is the same property `rw_variance_forecast` is tested for
in `tests/test_evaluation.py`.

Had the shift been skipped, $p_t$ would have been present in `h_garch_n`, which
would have scored the method better than it deserves.

## Scoring

**QLIKE loss.** For a variance proxy $p_t$ and forecast $h_t$,
$\ \mathrm{QLIKE}_t = \dfrac{p_t}{h_t} - \ln\dfrac{p_t}{h_t} - 1$.

**MSE loss.** $\mathrm{MSE}_t = (p_t - h_t)^2$.

Both are averaged over the test window. QLIKE is scale-invariant and penalizes
under-prediction more heavily, which is the asymmetry a risk manager wants
(Patton 2011).

`qlike` refuses $p \le 0$ by contract, so the zero-return days in the test window
are counted first, before any loss is computed.

In [7]:
# Q1: MSE will be dominated by large discrepancies because of the squared term.

# Q2: h=p/2 has the larger QLIKE, so under-prediction is punished harder than
# over-prediction. That is the conservative direction for risk management.

# Zero-return day count
zero_return_days = (r_test == 0).sum()
print(f"Number of zero-return days in the test set: {zero_return_days}")
# There are none, so no observation has to be dropped or clipped for QLIKE.

# Compute QLIKE for each model
qlike_garch_n = qlike(p, h_garch_n)
qlike_garch_t = qlike(p, h_garch_t)
qlike_rw = qlike(p, h_rw)
qlike_unc = qlike(p, h_unc)

# Compute MSE for each model
mse_garch_n = mse(p, h_garch_n)
mse_garch_t = mse(p, h_garch_t)
mse_rw = mse(p, h_rw)
mse_unc = mse(p, h_unc)

# Create a DataFrame to display the results
results = pd.DataFrame({
    "QLIKE": [qlike_garch_n, qlike_garch_t, qlike_rw, qlike_unc],
    "MSE": [mse_garch_n, mse_garch_t, mse_rw, mse_unc]
}, index=["GARCH(1,1)-N", "GARCH(1,1)-t", "Random Walk", "Unconditional"])

print("GARCH(1,1)-N vs Unconditional QLIKE (in %):", (qlike_garch_n - qlike_unc) / qlike_unc * 100)
print("GARCH(1,1)-N vs Random Walk QLIKE (in %):", (qlike_garch_n - qlike_rw) / qlike_rw * 100)
print("GARCH(1,1)-N vs Unconditional MSE (in %):", (mse_garch_n - mse_unc) / mse_unc * 100)
print("GARCH(1,1)-N vs Random Walk MSE (in %):", (mse_garch_n - mse_rw) / mse_rw * 100)

results

Number of zero-return days in the test set: 0
GARCH(1,1)-N vs Unconditional QLIKE (in %): -24.91071910502538
GARCH(1,1)-N vs Random Walk QLIKE (in %): -5.927970775941836
GARCH(1,1)-N vs Unconditional MSE (in %): -26.281728459697778
GARCH(1,1)-N vs Random Walk MSE (in %): -17.45052525090562


,QLIKE,MSE
"GARCH(1,1)-N",1.530756,34.646702
"GARCH(1,1)-t",1.535537,35.301223
Random Walk,1.627217,41.970833
Unconditional,2.038582,46.998799


The GARCH models beat both the random walk and the unconditional volatility on
QLIKE and on MSE, so they offer a better forecast of tomorrow's volatility. The
margin over the random walk is modest: we beat the naive persistence model, but
only by a little.

GARCH fitted with normal innovations is close to indistinguishable from the
Student-t version, though slightly better. The difference lies in the tails, so
the t's heavier tails are more heavily penalized by these metrics.

## The sober conclusion

Does GARCH(1,1) beat naive volatility forecasts out of sample?

Using S&P 500 daily log returns from 2005, with a 70/30 chronological split and
1,629 test days, we find that GARCH(1,1)-N gives a solid reduction of 24.9%
QLIKE and 26.3% MSE against unconditional volatility, and a modest reduction of
5.9% QLIKE and 17.5% MSE against a 21-day rolling variance random walk. These
are one-day-ahead variance forecasts scored against squared returns, with
parameters frozen at their training estimates. We are still missing the error
bars that would support the apparent positive verdict.

Our test window opens on 7 February 2020, so training is entirely pre-COVID,
which heavily penalizes the constant baseline. We fit both a normal and a
Student-t innovation distribution and found the Gaussian slightly better
(1.5308 vs 1.5355 QLIKE).

**Next steps**, each tied to something measured above:

- The leverage effect measured in notebook 01 is unmodelled, since GARCH(1,1) is
  sign-blind. GJR-GARCH or EGARCH would address it.
- The ACF of |r| decays more slowly than any exponential (fig1), which points to
  HAR-RV on true realized volatility.
- A single split with frozen parameters should become walk-forward
  re-estimation.
- The QLIKE gap carries no significance statement, which needs a
  Diebold-Mariano test with a HAC standard error.